# Calibration Analysis
Affine model calibration of CDS spreads: run calibration loop, evaluate RMSE/correlation metrics, and visualize fit quality across firms and leverage groups.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Check if a directory looks like the project root (has data/output and src)
def _is_project_root(p: Path) -> bool:
    return (p / 'src' / 'calibration' / 'affine_calibration.py').exists()

# Walk up from cwd to find the project root directory
def _find_project_root() -> Path:
    candidates = []

    # 1) Notebook file location if available
    try:
        nb_dir = Path(__file__).resolve().parent
        candidates.extend([nb_dir, *nb_dir.parents])
    except NameError:
        pass

    # 2) Current working directory and its parents
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    # 3) Common local workspace fallback for this project name
    home = Path.home()
    candidates.extend([
        home / 'Downloads' / 'Seminar QF' / 'Seminar QF',
        home / 'Downloads' / 'Seminar QF',
    ])

    # Deduplicate while preserving order
    seen = set()
    unique_candidates = []
    for c in candidates:
        c = c.resolve()
        if c not in seen:
            unique_candidates.append(c)
            seen.add(c)

    for c in unique_candidates:
        if _is_project_root(c):
            return c

    raise RuntimeError(
        'Could not locate project root containing src/calibration/affine_calibration.py. '
        'Open notebook from the project workspace or update root detection.'
    )

PROJECT_ROOT = _find_project_root()

# Put project root on sys.path so 'src' is importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Paths
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output'
INPUT_DIR = PROJECT_ROOT / 'data' / 'input'
MERTON_FILE = OUTPUT_DIR / 'merged_data_with_merton.csv'
CALIB_DIR = OUTPUT_DIR / 'calibration'
CALIB_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'src exists   : {(PROJECT_ROOT / "src").exists()}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Input dir    : {INPUT_DIR}')
print(f'Calibration  : {CALIB_DIR}')

In [ ]:
# Import the self-contained calibration module
# (reload to pick up any source edits made between runs)
import importlib, src.calibration.affine_calibration, src.utils.plots.calibration_plots
importlib.reload(src.calibration.affine_calibration)
importlib.reload(src.utils.plots.calibration_plots)

from src.calibration.affine_calibration import (
    calibrate_model,
    compute_metrics,
    compute_firm_level_metrics,
    DEFAULT_WINDOW_DAYS,
    MODEL_CONFIGS,
    MATURITIES,
)
from src.utils.plots.calibration_plots import (
    plot_timeseries,
    plot_scatter_comparison,
    plot_rolling_betas,
    plot_rmse_comparison,
    metrics_summary_table,
    plot_mean_median_vs_market,
    plot_mean_median_by_leverage,
)

print(f'Estimation window : {DEFAULT_WINDOW_DAYS} trading days  '
      f'(≈ {DEFAULT_WINDOW_DAYS // 5} weeks)')
print(f'Models available  : {list(MODEL_CONFIGS.keys())}')
print(f'Maturities        : {MATURITIES}')

In [ ]:
# Store all results and metrics
all_results = {}   # {model: {maturity: DataFrame}}
all_metrics = {}   # {model: {maturity: metrics_dict}}

for model_name in MODEL_CONFIGS:
    mc_file = OUTPUT_DIR / MODEL_CONFIGS[model_name]['mc_file']
    if not mc_file.exists():
        print(f'\n⚠  Skipping {model_name}: {mc_file.name} not found.')
        continue

    print(f'\n{"="*60}')
    print(f'  Calibrating: {model_name}')
    print(f'{"="*60}')

    results = calibrate_model(
        model_name=model_name,
        output_dir=OUTPUT_DIR,
        input_dir=INPUT_DIR,
        merton_file=MERTON_FILE,
        window=DEFAULT_WINDOW_DAYS,
        convert_model_to_bps=True,
    )
    all_results[model_name] = results

    # Compute aggregate metrics per maturity
    model_metrics = {}
    for mat_key, cal_df in results.items():
        model_metrics[mat_key] = compute_metrics(cal_df)
    all_metrics[model_name] = model_metrics

print('\n✓ Calibration complete for all available models.')

In [ ]:
summary = metrics_summary_table(all_metrics)

# Display a nicely formatted table
display_cols = [
    'Model', 'Maturity',
    'rmse_raw', 'rmse_calibrated',
    'mae_raw', 'mae_calibrated',
    'corr_levels_raw', 'corr_levels_cal',
    'corr_changes_raw', 'corr_changes_cal',
    'mean_beta0', 'mean_beta1',
]
display_cols = [c for c in display_cols if c in summary.columns]
summary_display = summary[display_cols].copy()

# Round for readability
for c in summary_display.columns:
    if summary_display[c].dtype in ['float64', 'float32']:
        summary_display[c] = summary_display[c].round(4)

print('Calibration Metrics Summary')
print('=' * 80)
display(summary_display)

# Save to CSV
summary_display.to_csv(CALIB_DIR / 'calibration_metrics_summary.csv', index=False)
print(f'\nSaved to {CALIB_DIR / "calibration_metrics_summary.csv"}')

## Calibration Plots

In [ ]:
plot_rmse_comparison(
    all_metrics,
    save_path=str(CALIB_DIR / 'rmse_raw_vs_calibrated.png'),
)

In [ ]:
for model_name, results in all_results.items():
    for mat_key in ['5y', '3y', '1y']:
        if mat_key in results:
            plot_scatter_comparison(
                results[mat_key],
                model_name=model_name,
                maturity=mat_key,
                save_path=str(CALIB_DIR / f'scatter_{model_name.lower().replace("-", "_")}_{mat_key}.png'),
            )

In [ ]:
# Pick the first available model and the 5y maturity for illustration
demo_model = list(all_results.keys())[0]
demo_mat   = '5y'
demo_df    = all_results[demo_model][demo_mat]

# Select up to 4 firms with the most calibrated observations
firm_counts = demo_df.dropna(subset=['calibrated']).groupby('gvkey').size().sort_values(ascending=False)
top_firms   = firm_counts.head(4).index.tolist()

print(f'Showing {demo_model} {demo_mat} for firms: {top_firms}\n')

for gvkey in top_firms:
    plot_timeseries(
        demo_df, gvkey,
        title=f'{demo_model} {demo_mat} – Firm {gvkey}',
        save_path=str(CALIB_DIR / f'timeseries_{demo_model.lower().replace("-","_")}_{demo_mat}_{gvkey}.png'),
    )

In [ ]:
for model_name, results in all_results.items():
    for mat_key in ['5y']:
        if mat_key in results:
            plot_rolling_betas(
                results[mat_key],
                model_name=model_name,
                maturity=mat_key,
                save_path=str(CALIB_DIR / f'betas_{model_name.lower().replace("-","_")}_{mat_key}.png'),
            )

In [ ]:
for model_name, results in all_results.items():
    if '5y' not in results:
        continue
    firm_metrics = compute_firm_level_metrics(results['5y'])
    firm_metrics = firm_metrics.sort_values('rmse_calibrated')

    print(f'\n{"="*60}')
    print(f'  {model_name} – 5Y Firm-Level Metrics')
    print(f'{"="*60}')

    display_cols_firm = ['company', 'n_obs', 'rmse_raw', 'rmse_calibrated',
                         'corr_levels_raw', 'corr_levels_cal',
                         'corr_changes_raw', 'corr_changes_cal']
    display_cols_firm = [c for c in display_cols_firm if c in firm_metrics.columns]
    display(firm_metrics[display_cols_firm].round(4))

    # Save
    fname = f'firm_metrics_{model_name.lower().replace("-","_")}_5y.csv'
    firm_metrics.to_csv(CALIB_DIR / fname, index=False)
    print(f'Saved to {CALIB_DIR / fname}')

In [ ]:
for model_name, results in all_results.items():
    for mat_key, cal_df in results.items():
        fname = f'calibrated_spreads_{model_name.lower().replace("-","_")}_{mat_key}.csv'
        cal_df.to_csv(CALIB_DIR / fname, index=False)
        n = cal_df['calibrated'].notna().sum()
        print(f'  {model_name} {mat_key}: saved {n:,} calibrated obs → {fname}')

print(f'\nAll calibrated spreads saved to {CALIB_DIR}')

In [ ]:
for model_name, results in all_results.items():
    for mat_key in ['1y', '3y', '5y']:
        if mat_key in results:
            plot_mean_median_vs_market(
                results[mat_key],
                model_name=model_name,
                maturity=mat_key,
                save_path=str(CALIB_DIR / f'mean_median_{model_name.lower().replace("-","_")}_{mat_key}.png'),
            )

In [ ]:
# MERTON_FILE is already defined in the path-setup cell above

for model_name, results in all_results.items():
    for mat_key in ['5y', '3y', '1y']:
        if mat_key not in results:
            continue
        for agg in ['mean', 'median']:
            plot_mean_median_by_leverage(
                results[mat_key],
                merton_file=str(MERTON_FILE),
                model_name=model_name,
                maturity=mat_key,
                leverage_metric='debt_to_equity',
                n_groups=3,
                aggregation=agg,
                save_path=str(CALIB_DIR / f'leverage_{agg}_{model_name.lower().replace("-","_")}_{mat_key}.png'),
            )